# Pythia longitudinal CPS sweep

A single checkpoint gives a local portrait. This notebook turns CPS into a developmental instrument by repeating the same governed probe across a checkpoint sequence.

## Scientific question

Does coupling fragility evolve smoothly with training, appear at identifiable transitions, or fluctuate at the level of batch and projection noise?

The default grid is intentionally small. Set `CPS_REVISIONS` to a comma-separated sequence for a longer campaign. Every checkpoint receives an independent manifest.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import show_environment
runtime = show_environment()

In [ ]:
import os
from cps.notebook import show_config
from cps.pythia.config import load_probe_config

revisions = [item.strip() for item in os.environ.get(
    "CPS_REVISIONS", "step0,step1,step16,step512,step1000"
).split(",") if item.strip()]
config = load_probe_config("subjects/pythia/configs/pythia_70m_longitudinal.yaml")
show_config(config)
print("[CAMPAIGN] checkpoint order:", revisions, flush=True)

## Execute the longitudinal campaign

Each child probe emits its own stage-by-stage log. The outer runner additionally reports checkpoint completion so a remote Colab log reveals exactly where a campaign stopped.

In [ ]:
from cps.pythia.runner import run_longitudinal

summary_path = run_longitudinal(config, revisions)
print(f"[CAMPAIGN] summary={summary_path}", flush=True)

## Compare checkpoints

The table below places projection quality and phase-envelope risk on the same row. A rising risk measure is not meaningful if closure quality is simultaneously degrading.

In [ ]:
import matplotlib.pyplot as plt
from cps.notebook import display_longitudinal_summary

frame = display_longitudinal_summary(summary_path)
if len(frame):
    axis = frame.plot(
        x="revision",
        y=["max phase spectral radius", "max finite-horizon gain"],
        marker="o",
        figsize=(10, 5),
    )
    axis.set_title("Longitudinal CPS observables")
    axis.set_ylabel("diagnostic value")
    axis.grid(True, alpha=0.25)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Next inference step

A longitudinal curve becomes predictive evidence only when thresholds and horizons are fixed before inspecting future loss or gradient events. Use the feature and prediction modules to perform grouped, prospective evaluation rather than selecting a compelling checkpoint after the fact.

In [ ]:
from cps.notebook import export_artifacts
archive = export_artifacts()
print(f"Artifact archive ready for colab-cli download: {archive}", flush=True)